# Agente Simples do Semantic Kernel com plugin de SQL
Agente do Semantic Kernel que se conecta ao azure foundry e usa um plugin para se conectar ao lakehouse endpoint para que possa ler os dados.   

Para realizar a tarefa será necessário:  
- Conexão com o Lakehouse via PyOdbc.
- **Usar um service principal** para a conexão sem o Entra ID para facilitar a autenticação sem necessitar de interatividade. Depois de criar o service principal de as permissoes para que ele possa ler os dados do Lakehouse, sem isso ocorrerá um erro.  
- Variáveis de ambiente em um arquivo `.env` para simplificar os testes e aprendizado.  
- AI Foundry com alguns modelos de LLM ou SLM (Para este caso pode ser qualquer um). 
- Framework UV instalada para gerenciamento de bibliotecas.
- Conta da Azure. 

Para realizar a conexão existem duas formas, com somente o PYODBC e usando o PYODBC + SQLALCHEMY.  
O uso fica  a seu critério, abaixo tem um exemplo com as duas formas. 

## Sobre a query 

A query utilizada no exemplo abaixo serve para exemplificar como pegar um sample com 10 linhas , fique a vontade para poder modificar como desejar.  

Ele randomiza os dados no nível do banco de dados e busca apenas o número exato de linhas necessárias, minimizando o tráfego de rede e o uso de memória.  

A função NEWID() gera um identificador único para cada linha, e ordená-las por esse identificador randomiza efetivamente o conjunto de resultados antes de aplicar o limite TOP (10).  
``` sql
SELECT TOP (10) [AccountKey],
			[ParentAccountKey],
			[AccountCodeAlternateKey],
			[ParentAccountCodeAlternateKey],
			[AccountDescription],
			[AccountType],
			[Operator],
			[CustomMembers],
			[ValueType],
			[CustomMemberOptions]
FROM [MyAdvWorksLH].[dbo].[dimaccount]
ORDER BY NEWID()
```
## Referencias
- [Connect to Microsoft Fabric Warehouse using Python and SQLAlchemy](https://medium.com/@mariusz_kujawski/connect-to-microsoft-fabric-warehouse-using-python-and-sqlalchemy-1e1179855037)
- [Connect to Fabric Lakehouses & Warehouses from Python code](https://debruyn.dev/2023/connect-to-fabric-lakehouses-warehouses-from-python-code/)
- [Microsoft Fabric and Langchain SQL Integration for Natural Language to SQL](https://blog.gopenai.com/microsoft-fabric-and-langchain-sql-integration-for-natural-language-to-sql-51b448836017)
- [Criar um service Principal e se conectar no Fabric](https://youtu.be/IFp1Aingnmw)

# Agenda 
Será dividido em duas partes, a primeira onde será mostrado como se conectar ao Fabric Usando o Pyodbc e a segunda onde será criado um agente simples como Semantic Kernel para que se conecte ao plugin (tool) que se conecta ao Fabric com a query gerada.  

1. Conexão PyODBC e SQLALCHEMY
2. Agente Simples que gera NL2SQL 
3. Multiagentes que se conectam ao Lakehouse e retorna um relatorio a partir da pergunta do usuário

Entendendo como essa conexão funciona fica fácil de realizar modificações para que atenda a diversos casos de uso.  
Exemplos: 
- Multiagentes para criar relatorios
- Multiplos Agentes em um fluxo com guardrails extras
- Adição do chat do agente a um dashboard do PowerBI



# Parte 1 - Conexão ao Fabric Lakehouse



## Importando as Bibliotecas a serem usadas

In [1]:
# Instalar e importar dependências
import os, pyodbc, struct, urllib
from dotenv import load_dotenv, dotenv_values 
from azure.identity import ClientSecretCredential
from itertools import chain, repeat
import pandas as pd
import sqlalchemy as sa

# Carregar variáveis do arquivo .env de um diretório específico
env_path = "../../../.env"
load_dotenv(dotenv_path=env_path)


True

## Criando a conexão

In [ ]:
# Crie um service principal no Microsift Entra e obtenha as credenciais
# Referencia: https://youtu.be/IFp1Aingnmw

tenant_id = os.getenv("TENANT_ID")
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

if not all([tenant_id, client_id, client_secret]):
    raise ValueError("Por favor, configure as variáveis TENANT_ID, CLIENT_ID e CLIENT_SECRET no arquivo .env")

# Conecta inicializa o ClientSecretCredential
credential = ClientSecretCredential(tenant_id, client_id, client_secret)
# Esta é a URL do recurso para o qual você está solicitando um token. Para SQL Databases, geralmente é "https://database.windows.net/.default"
# Você pode precisar ajustar isso dependendo do serviço específico que está acessando.
resource_url = "https://database.windows.net/.default"
# Recupera um token de acesso válido para se conectar a bancos de dados SQL
token_object = credential.get_token(resource_url)
# Definicao da Connection String
# O endpoint do SQL Server no Azure Fabric
sql_endpoint = os.getenv("SQL_ENDPOINT")
# Nome do banco de dados que você deseja acessar
database = os.getenv("LAKEHOUSE")
# Cria a string de conexão com o driver ODBC
connection_string = f"Driver={{ODBC Driver 18 for SQL Server}};Server={sql_endpoint},1433;Database=f{database};Encrypt=Yes;TrustServerCertificate=No"
# Adiciona o token de acesso ao dicionário de atributos
params = urllib.parse.quote(connection_string)


## Usando o SQLAlchemy com Pandas

In [3]:
# Prepara o token de acesso para ser usado na conexão
token_as_bytes = bytes(token_object.token, "UTF-8") # Converte o token para uma string UTF-8
encoded_bytes = bytes(chain.from_iterable(zip(token_as_bytes, repeat(0)))) # decodifica o token para bytes, adicionando um byte de preenchimento a cada byte do token
token_bytes = struct.pack("<i", len(encoded_bytes)) + encoded_bytes # empacota o token em um formato binário, prefixando com o tamanho do token como um inteiro de 4 bytes
attrs_before = {1256: token_bytes}  # Define os atributos de conexão, onde 1256 é o código de atributo para o token de acesso no driver ODBC do SQL Server
# Cria a engine do SQLAlchemy com a string de conexão e os atributos de conexão
engine = sa.create_engine("mssql+pyodbc:///?odbc_connect={0}".format(params), connect_args={'attrs_before': attrs_before})

SQL_QUERY = """SELECT TOP (10) [AccountKey],
			[ParentAccountKey],
			[AccountCodeAlternateKey],
			[ParentAccountCodeAlternateKey],
			[AccountDescription],
			[AccountType],
			[Operator],
			[CustomMembers],
			[ValueType],
			[CustomMemberOptions]
FROM [MyAdvWorksLH].[dbo].[dimaccount]
ORDER BY NEWID()
"""

# Executa a consulta SQL e armazena o resultado em um DataFrame do Pandas
df = pd.read_sql(SQL_QUERY, engine)
df.head()  # Exibe as primeiras linhas do DataFrame para verificação

,AccountKey,ParentAccountKey,AccountCodeAlternateKey,ParentAccountCodeAlternateKey,AccountDescription,AccountType,Operator,CustomMembers,ValueType,CustomMemberOptions
0,61,59,6020,600,Payroll Taxes,Expenditures,+,None,Currency,None
1,20,17,1230,1200,Machinery & Equipment,Assets,+,None,Currency,None
2,74,58,6500,60,Professional Services,Expenditures,+,None,Currency,None
3,22,17,1250,1200,Leasehold Improvements,Assets,+,None,Currency,None
4,18,17,1210,1200,Land & Improvements,Assets,+,None,Currency,None


## Usando Somente o PyODBC

In [4]:
# Usando somente o PyODBC para conexão e consulta
connection = pyodbc.connect(connection_string, attrs_before=attrs_before)

# cria um cursor para executar consultas
cursor = connection.cursor()

# Executa a consulta SQL
cursor.execute(SQL_QUERY)

# Busca e imprime 10 linhas uma a uma
rows = cursor.fetchmany(10)
for row in rows:
    print(row)

# Encerra o cursor e a conexão
cursor.close()
connection.close()

(94, 47, 8500, 4, 'Taxes', 'Expenditures', '-', None, 'Currency', None)
(1, None, 1, None, 'Balance Sheet', None, '~', None, 'Currency', None)
(64, 58, 620, 60, 'Travel Expenses', 'Expenditures', '+', None, 'Currency', None)
(92, 88, 8030, 80, 'Other Income', 'Revenue', '+', None, 'Currency', None)
(101, 51, 4200, 4110, 'Trade Sales', 'Revenue', '+', None, 'Currency', None)
(45, 44, 3540, 3030, 'Prior Year Retained Earnings', 'Liabilities', '+', None, 'Currency', None)
(11, 9, 1164, 1160, 'Work in Process', 'Assets', '+', None, 'Currency', None)
(81, 79, 6820, 680, 'Vehicles', 'Expenditures', '+', None, 'Currency', None)
(52, 51, 4500, 4110, 'Intercompany Sales', 'Revenue', '+', None, 'Currency', None)
(18, 17, 1210, 1200, 'Land & Improvements', 'Assets', '+', None, 'Currency', None)


## Conclusão parte 1
Aqui realizamos as seguintes tarefas: 
- Criamos uma conexão com o Lakehouse
- Executamos uma query
- Lemos os resultados em Pandas e PyOdbc nativo.

# Parte 2 - Criação do Agente com o Semantic Kernel

## Schema das tabelas do Lakehouse

Existem algumas formas de saber qual o esquema das tabelas do Lakehouse.
Uma delas é executar a seguinte query: 
``` sql
SELECT 
    t.name AS TableName,
    c.name AS ColumnName,
    tp.name AS DataType
FROM 
    sys.tables AS t
INNER JOIN 
    sys.columns AS c ON t.object_id = c.object_id
INNER JOIN 
    sys.types AS tp ON c.user_type_id = tp.user_type_id
ORDER BY 
    t.name, c.column_id;
```
Outra opção é pegar o CSV de exemplo no caminho: `MS-Fabric-Workshop\assets\data\AdvWorksDatasets\TablesMetadata.csv` e usar ele como instruções para o agente saber a estrutura das tabelas para query. 


## Exemplos de Queries

1. Receita total por ano fiscal  
``` sql
SELECT   
    d.CalendarYear,  
    SUM(fis.SalesAmount) AS TotalSalesAmount  
FROM factinternetsales fis  
JOIN dimdate d ON fis.OrderDateKey = d.DateKey  
GROUP BY d.CalendarYear  
ORDER BY d.CalendarYear;  
```

2. Receita mensal por categoria de produto (Ano atual -15 dados nao estao atualizados)  
``` sql
WITH Last12MonthsDates AS (  
    SELECT DateKey  
    FROM dimdate  
    WHERE FullDateAlternateKey >= DATEADD(YEAR, -15, CAST(GETDATE() AS DATE))  
)  
SELECT   
    d.CalendarYear,  
    d.MonthNumberOfYear,  
    pc.EnglishProductCategoryName,  
    SUM(fis.SalesAmount) AS TotalSales  
FROM factinternetsales fis  
INNER JOIN dimdate d ON fis.OrderDateKey = d.DateKey  
INNER JOIN dimproduct p ON fis.ProductKey = p.ProductKey  
INNER JOIN dimproductsubcategory psc ON p.ProductSubcategoryKey = psc.ProductSubcategoryKey  
INNER JOIN dimproductcategory pc ON psc.ProductCategoryKey = pc.ProductCategoryKey  
WHERE d.DateKey IN (SELECT DateKey FROM Last12MonthsDates)  
GROUP BY   
    d.CalendarYear,  
    d.MonthNumberOfYear,  
    pc.EnglishProductCategoryName  
ORDER BY   
    d.CalendarYear,   
    d.MonthNumberOfYear,   
    TotalSales DESC;   
```

3. Clientes com maior receita acumulada 
``` sql 
SELECT TOP 10  
    c.CustomerKey,  
    c.FirstName,  
    c.LastName,  
    SUM(fis.SalesAmount) AS TotalCustomerSales  
FROM factinternetsales fis  
JOIN dimcustomer c ON fis.CustomerKey = c.CustomerKey  
GROUP BY c.CustomerKey, c.FirstName, c.LastName  
ORDER BY TotalCustomerSales DESC;  
```

4. Análise de receita por região de vendas (território)  
``` sql
SELECT   
    st.SalesTerritoryRegion,  
    SUM(fis.SalesAmount) AS RegionSales  
FROM factinternetsales fis  
JOIN dimsalesterritory st ON fis.SalesTerritoryKey = st.SalesTerritoryKey  
GROUP BY st.SalesTerritoryRegion  
ORDER BY RegionSales DESC;  
```

5. Receita média por pedido por funcionário (vendedor)  
``` sql
SELECT  
    e.EmployeeKey,  
    e.FirstName,  
    e.LastName,  
    AVG(frs.SalesAmount) AS AvgSalesPerOrder  
FROM factresellersales frs  
JOIN dimemployee e ON frs.EmployeeKey = e.EmployeeKey  
GROUP BY e.EmployeeKey, e.FirstName, e.LastName  
ORDER BY AvgSalesPerOrder DESC;  
```

6. Volume total de vendas e descontos concedidos por promoção
``` sql  
SELECT   
    p.EnglishPromotionName,  
    COUNT(fis.SalesOrderNumber) AS NumberOfOrders,  
    SUM(fis.SalesAmount) AS TotalSalesAmount,  
    SUM(fis.DiscountAmount) AS TotalDiscountAmount  
FROM factinternetsales fis  
JOIN dimpromotion p ON fis.PromotionKey = p.PromotionKey  
GROUP BY p.EnglishPromotionName  
ORDER BY TotalSalesAmount DESC;  
```
  
7. Produtos mais vendidos por quantidade no último trimestre fiscal  
``` sql
WITH MaxSaleDate AS (  
    SELECT MAX(OrderDateKey) AS MaxOrderDateKey  
    FROM factinternetsales  
),  
LatestFiscalPeriod AS (  
    SELECT   
        d.FiscalYear,   
        d.FiscalQuarter  
    FROM dimdate d  
    CROSS JOIN MaxSaleDate m  
    WHERE d.DateKey = m.MaxOrderDateKey  
)  
SELECT TOP 10  
    p.EnglishProductName,  
    SUM(fis.OrderQuantity) AS TotalQuantitySold  
FROM factinternetsales fis  
INNER JOIN dimproduct p ON fis.ProductKey = p.ProductKey  
INNER JOIN dimdate d ON fis.OrderDateKey = d.DateKey  
CROSS JOIN LatestFiscalPeriod lfp  
WHERE d.FiscalYear = lfp.FiscalYear   
  AND d.FiscalQuarter = lfp.FiscalQuarter  
GROUP BY p.EnglishProductName  
ORDER BY TotalQuantitySold DESC;  
```  
8. Comparação da receita total entre canais de venda: Internet vs Revendedores (Reseller)  
``` sql
WITH InternetSales AS (  
    SELECT   
        SUM(SalesAmount) AS TotalInternetSales  
    FROM factinternetsales  
),  
ResellerSales AS (  
    SELECT  
        SUM(SalesAmount) AS TotalResellerSales  
    FROM factresellersales  
)  
SELECT   
    'Internet' AS SalesChannel,  
    TotalInternetSales AS TotalSalesAmount  
FROM InternetSales  
UNION ALL  
SELECT   
    'Reseller' AS SalesChannel,  
    TotalResellerSales AS TotalSalesAmount  
FROM ResellerSales;  
```
9. Análise mensal da receita por segmento de cliente (baseada em YearlyIncome)  
``` sql
WITH CustomerSegment AS (  
    SELECT   
        CustomerKey,  
        CASE   
            WHEN YearlyIncome < 30000 THEN 'Low Income'  
            WHEN YearlyIncome BETWEEN 30000 AND 70000 THEN 'Mid Income'  
            ELSE 'High Income'  
        END AS IncomeSegment  
    FROM dimcustomer  
), SalesWithSegment AS (  
    SELECT  
        d.CalendarYear,  
        d.MonthNumberOfYear,  
        cs.IncomeSegment,  
        fis.SalesAmount  
    FROM factinternetsales fis  
    JOIN dimdate d ON fis.OrderDateKey = d.DateKey  
    JOIN CustomerSegment cs ON fis.CustomerKey = cs.CustomerKey  
    WHERE d.FullDateAlternateKey >= DATEADD(YEAR, -12, GETDATE())  
)  
SELECT  
    CalendarYear,  
    MonthNumberOfYear,  
    IncomeSegment,  
    SUM(SalesAmount) AS TotalSales  
FROM SalesWithSegment  
GROUP BY CalendarYear, MonthNumberOfYear, IncomeSegment  
ORDER BY CalendarYear, MonthNumberOfYear, IncomeSegment;  
```

10. Identificar funcionários com maior quota de vendas realizadas vs. quota definida
``` sql 
WITH SalesByEmployee AS (  
    SELECT   
        e.EmployeeKey,  
        e.FirstName,  
        e.LastName,  
        SUM(frs.SalesAmount) AS ActualSales  
    FROM factresellersales frs  
    INNER JOIN dimemployee e ON frs.EmployeeKey = e.EmployeeKey  
    GROUP BY e.EmployeeKey, e.FirstName, e.LastName  
),  
QuotaByEmployee AS (  
    SELECT   
        EmployeeKey,  
        SUM(SalesAmountQuota) AS SalesQuota  
    FROM factsalesquota  
    GROUP BY EmployeeKey  
)  
SELECT   
    s.EmployeeKey,  
    s.FirstName,  
    s.LastName,  
    s.ActualSales,  
    ISNULL(q.SalesQuota, 0) AS SalesQuota,  
    CASE   
        WHEN ISNULL(q.SalesQuota, 0) = 0 THEN NULL  
        ELSE CAST(s.ActualSales * 100.0 / q.SalesQuota AS DECIMAL(5,2))  
    END AS PercentOfQuotaAchieved  
FROM SalesByEmployee s  
LEFT JOIN QuotaByEmployee q ON s.EmployeeKey = q.EmployeeKey  
ORDER BY   
    CASE WHEN q.SalesQuota IS NULL OR q.SalesQuota = 0 THEN 1 ELSE 0 END ASC,  
    PercentOfQuotaAchieved DESC;   
```

11. Produtos com maior margem de lucro (SalesAmount - TotalProductCost)  
``` sql
SELECT TOP 20  
    p.EnglishProductName,  
    SUM(fis.SalesAmount) AS TotalSales,  
    SUM(fis.TotalProductCost) AS TotalCost,  
    SUM(fis.SalesAmount) - SUM(fis.TotalProductCost) AS TotalMargin,  
    CAST((SUM(fis.SalesAmount) - SUM(fis.TotalProductCost)) / NULLIF(SUM(fis.SalesAmount),0) * 100 AS DECIMAL(5,2)) AS MarginPct  
FROM factinternetsales fis  
JOIN dimproduct p ON fis.ProductKey = p.ProductKey  
GROUP BY p.EnglishProductName  
ORDER BY TotalMargin DESC;  
```

12. Análise de vendas por categoria e subcategoria com pivot para facilitar consumo em BI  
``` sql
SELECT  
    pc.EnglishProductCategoryName,  
    psc.EnglishProductSubcategoryName,  
    SUM(fis.SalesAmount) AS TotalSales  
FROM factinternetsales fis  
JOIN dimproduct p ON fis.ProductKey = p.ProductKey  
JOIN dimproductsubcategory psc ON p.ProductSubcategoryKey = psc.ProductSubcategoryKey  
JOIN dimproductcategory pc ON psc.ProductCategoryKey = pc.ProductCategoryKey  
GROUP BY pc.EnglishProductCategoryName, psc.EnglishProductSubcategoryName  
ORDER BY pc.EnglishProductCategoryName, TotalSales DESC;  
```

13. Tempo médio entre encomenda e envio por território de vendas  
``` sql
SELECT   
    st.SalesTerritoryRegion,  
    AVG(DATEDIFF(DAY, fis.OrderDate, fis.ShipDate)) AS AvgDaysToShip  
FROM factinternetsales fis  
JOIN dimsalesterritory st ON fis.SalesTerritoryKey = st.SalesTerritoryKey  
WHERE fis.ShipDate IS NOT NULL AND fis.OrderDate IS NOT NULL  
GROUP BY st.SalesTerritoryRegion  
ORDER BY AvgDaysToShip;  
```

14. Análise de receita por gênero de cliente nos últimos 2 anos  
``` sql
SELECT  
    c.Gender,  
    d.CalendarYear,  
    SUM(fis.SalesAmount) AS TotalSales  
FROM factinternetsales fis  
JOIN dimcustomer c ON fis.CustomerKey = c.CustomerKey  
JOIN dimdate d ON fis.OrderDateKey = d.DateKey  
WHERE d.FullDateAlternateKey >= DATEADD(YEAR, -12, GETDATE())  
GROUP BY c.Gender, d.CalendarYear  
ORDER BY d.CalendarYear, c.Gender;  
```

15. Ranking de clientes por frequência de pedidos e valor total gasto 
``` sql 
WITH CustomerOrders AS (  
    SELECT  
        fis.CustomerKey,  
        COUNT(DISTINCT fis.SalesOrderNumber) AS OrderCount,  
        SUM(fis.SalesAmount) AS TotalSpent  
    FROM factinternetsales fis  
    GROUP BY fis.CustomerKey  
)  
SELECT  
    co.CustomerKey,  
    c.FirstName,  
    c.LastName,  
    co.OrderCount,  
    co.TotalSpent,  
    RANK() OVER (ORDER BY co.TotalSpent DESC, co.OrderCount DESC) AS RankBySpend  
FROM CustomerOrders co  
JOIN dimcustomer c ON co.CustomerKey = c.CustomerKey  
ORDER BY RankBySpend; 
```


## Importação das Bibliotecas

In [5]:
import asyncio
from typing import Annotated

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions import kernel_function

## Criação do Kernel e Conexão com o AI Foundry

**Nosso primeiro Agente**  
Agora, vamos construir um agente simples para começar. Usaremos o `ChatCompletionAgent`, que abstrai a implementação subjacente. Em seguida, passamos o `AzureChatCompletion()`, que conecta o agente ao Azure OpenAI. Essa classe lê automaticamente as variáveis de ambiente padrão via o arquivo `.env` na raiz deste repositório:

- **AZURE_OPENAI_ENDPOINT** – O endpoint com o qual ele deve se comunicar por padrão  
- **AZURE_OPENAI_API_KEY** – A chave de API que deve ser usada  
- **AZURE_OPENAI_API_VERSION** – A versão da API de inferência que deve ser usada por padrão  
- **AZURE_OPENAI_CHAT_DEPLOYMENT_NAME** – O nome da implantação do modelo de chat que deve ser usado por padrão  
- **AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME** – O nome da implantação de embeddings que deve ser usado por padrão  

Obviamente, você pode sobrescrever manualmente qualquer um dos parâmetros do `AzureChatCompletion()`.

Por fim, podemos aguardar a resposta. Observe que não temos um histórico de conversa, então cada mensagem enviada ao agente é tratada como uma nova conversa.

In [6]:
# Para Chamar um agente simples 

simple_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="ai_nl2sql_assistant",
    instructions="""Você é um especialista em T-SQL, análise de dados e BI e deve gerar scripts T-SQL otimizados para execução no Lakehouse Endpoint do Microsoft Fabric.

# Detalhes da Função

- **Objetivo Principal**: O script gerado responderá dúvidas específicas, realizará análises, ou manipulará/extrairá dados com base nos requisitos do usuário.
- **Plataforma de Execução**: Compatível com Microsoft Fabric Lakehouse Endpoint (Synapse SQL/Databricks SQL).
- **Boas Práticas**:
  - Inclua otimizações para desempenho e uso eficiente dos recursos do Microsoft Fabric.
  - Siga padrões modernos de T-SQL para legibilidade e eficiência.
  - Garanta que as consultas sejam escaláveis e generalizáveis.
  - Se necessário, suporte dinâmicas que adaptem a consulta a diferentes parâmetros fornecidos.
- **Restrições de Data**: Ao trabalhar com tabelas contendo colunas de data, considere explicitamente a lógica abaixo:
    - Utilize a última data disponível na `dimdate` com base na tabela de fatos fornecida:
    ```sql
    SELECT MAX(FullDateAlternateKey) 
    FROM dimdate 
    WHERE FullDateAlternateKey <= (
        SELECT MAX(FullDateAlternateKey) 
        FROM factinternetsales
    )
    ```

# Requisitos para Formatação de Saída

1. O script gerado deve ser estruturado no seguinte formato de JSON:
   - **Chave `consulta`**: Contém a consulta SQL.
   - Exemplo de saída:
     ```json
     { "consulta": "Query SQL Gerada" }
     ```

2. A consulta deve ser encapsulada e otimizada para seu propósito final, garantindo usabilidade imediata.

# Passos Suplementares

- Valide os parâmetros e o contexto fornecido pelo usuário.
- Ajuste dinamicamente as condições ou filtros, se necessário.
- Se dados adicionais estruturais (como nomes de colunas/tabelas não especificados) forem necessários, destaque explicitamente como placeholders na saída.
- Explique casos de complexidade mais alta, se houver, através da estrutura do SQL.

# Output Format

A saída deverá ser fornecida em formato JSON:
```json
{ 
  "consulta": "Query SQL Gerada" 
}
```

# Exemplo de Uso

**Entrada do Usuário**:
"Preciso de um relatório das vendas totais filtradas pelas últimas datas disponíveis, sumarizando as vendas por categoria de produto e por país."

**Saída Esperada**:
```json
{
  "consulta": "SELECT pc.EnglishProductCategoryName AS ProductCategory, r.SalesTerritoryCountry AS Country, SUM(fs.SalesAmount) AS TotalSales FROM factinternetsales fs JOIN dimproduct p ON fs.ProductKey = p.ProductKey JOIN dimproductcategory pc ON p.ProductCategoryKey = pc.ProductCategoryKey JOIN dimsalesterritory r ON fs.SalesTerritoryKey = r.SalesTerritoryKey WHERE fs.OrderDateKey = (SELECT MAX(FullDateAlternateKey) FROM dimdate WHERE FullDateAlternateKey <= (SELECT MAX(FullDateAlternateKey) FROM factinternetsales)) GROUP BY pc.EnglishProductCategoryName, r.SalesTerritoryCountry ORDER BY TotalSales DESC"
}
```

# Notas

- Utilize aliases bem definidos para facilitar a leitura e evitar ambiguidades.
- Antecipe e otimize casos de tabelas grandes, garantindo eficiência em joins e agregações.
- Se filtros, colunas ou tabelas não forem explicitamente especificados, use placeholders correspondentes (`[TableName]`, `[ColumnName]`, etc.).
- Documente complexidades ou suposições na consulta para facilitar ajustes futuros."""
)


response = await simple_agent.get_response(messages="Quais são as vendas totais por categoria de produto considerando a última data disponível?")
print("Agent:", response.content)

Agent: ```json
{
  "consulta": "
    SELECT 
      pc.EnglishProductCategoryName AS ProductCategory,
      SUM(fs.SalesAmount) AS TotalSales
    FROM factinternetsales fs
    INNER JOIN dimproduct p ON fs.ProductKey = p.ProductKey
    INNER JOIN dimproductcategory pc ON p.ProductCategoryKey = pc.ProductCategoryKey
    WHERE fs.OrderDateKey = (
      SELECT MAX(FullDateAlternateKey)
      FROM dimdate
      WHERE FullDateAlternateKey <= (
        SELECT MAX(OrderDateKey) FROM factinternetsales
      )
    )
    GROUP BY pc.EnglishProductCategoryName
    ORDER BY TotalSales DESC
  "
}
```


Obs.: 
Note que baseado no que eu adicionei para as instruções do agente ele me traz o output esperado

## Exemplos de perguntas
Abaixo alguns exemplos de perguntas que podemos usar:  

- Quais são os 10 produtos com maior receita total de vendas, incluindo o nome do produto?
- Qual o total de vendas por ano e categoria de produto?
- Liste os funcionários que possuem metas de venda acima de um valor específico para o ano selecionado.
- Qual a receita total por território de vendas e tipo de promoção  para um período determinado?
- Mostre a quantidade total de unidades vendidas e em estoque  para os produtos agrupados por subcategoria .
- Quais clientes têm maior valor de compras totais e quais são seus dados demográficos principais?
- Apresente a média da taxa de câmbio por moeda  ao longo de um intervalo de datas especificado.

## Criação do Plugin/Function (Tool)

**Chamada de Funções e Plugins**  
Uma das funcionalidades mais poderosas dos agentes do Semantic Kernel é a capacidade de usar *plugins* e chamar *funções*. Isso permite que os agentes vão além da simples geração de texto e realmente executem ações, recuperem informações e se integrem a sistemas externos.

### O que são Plugins no Semantic Kernel?  
No Semantic Kernel, um *plugin* é uma coleção de funções relacionadas que podem ser registradas no *kernel* e disponibilizadas para os agentes. Os plugins podem:

- Recuperar informações de bancos de dados ou APIs  
- Realizar cálulos ou transformações de dados  
- Executar operações no sistema  
- Interagir com serviços externos  

Os plugins contêm uma ou mais funções, que podem ser:

- **Funções Nativas**: Escritas em código (Python) e executadas diretamente quando chamadas  
- **Funções Semânticas**: Definidas por *prompts* que são enviados ao LLM quando chamadas  

Vamos criar um plugin simples com funções nativas para demonstrar como isso funciona.

Até agora, tínhamos apenas um agente básico. Agora, vamos dar ao agente algumas ferramentas, que são chamadas de *Plugins* no Semantic Kernel. Com isso, podemos integrar facilmente com serviços externos e APIs.


In [ ]:
import json
import pyodbc
import struct
from datetime import datetime
from typing import Annotated, List, Dict, Any
from azure.identity import ClientSecretCredential
from itertools import chain, repeat

class FabricDatabasePlugin:
    """Plugin para interagir com o Microsoft Fabric Lakehouse via PyODBC."""
    
    def __init__(self):
        # Configuração da conexão usando as variáveis de ambiente já carregadas
        self.tenant_id = os.getenv("TENANT_ID")
        self.client_id = os.getenv("CLIENT_ID")
        self.client_secret = os.getenv("CLIENT_SECRET")
        self.sql_endpoint = os.getenv("SQL_ENDPOINT")
        self.database = os.getenv("LAKEHOUSE")
        self._setup_connection()
        
        # Cache para metadados
        self._metadata_cache = None
    
    def _setup_connection(self):
        """Configura a conexão com o Fabric Lakehouse."""
        if not all([self.tenant_id, self.client_id, self.client_secret]):
            raise ValueError("Credenciais não configuradas no .env")
        
        # Inicializa credentials
        self.credential = ClientSecretCredential(
            self.tenant_id, self.client_id, self.client_secret
        )
        
        # URL do recurso para token
        self.resource_url = "https://database.windows.net/.default"
        
        # String de conexão
        self.connection_string = (
            f"Driver={{ODBC Driver 18 for SQL Server}};"
            f"Server={self.sql_endpoint},1433;"
            f"Database={self.database};"
            f"Encrypt=Yes;"
            f"TrustServerCertificate=No"
        )

    def _get_connection_attributes(self):
        """Obtém os atributos de conexão com token de acesso."""
        token_object = self.credential.get_token(self.resource_url)
        token_as_bytes = bytes(token_object.token, "UTF-8")
        encoded_bytes = bytes(chain.from_iterable(zip(token_as_bytes, repeat(0))))
        token_bytes = struct.pack("<i", len(encoded_bytes)) + encoded_bytes
        return {1256: token_bytes}

    def _convert_value(self, value):
        """Converte valores para tipos serializáveis JSON."""
        if value is None:
            return None
        elif isinstance(value, (datetime, )):
            return value.isoformat()
        elif isinstance(value, (bytes, bytearray)):
            return f"<binary data: {len(value)} bytes>"
        elif hasattr(value, '__str__'):
            return str(value)
        else:
            return value

    def _execute_query_pyodbc(self, query: str) -> Dict[str, Any]:
        """Executa query usando apenas PyODBC e retorna resultado estruturado."""
        attrs_before = self._get_connection_attributes()
        connection = None
        cursor = None
        
        try:
            connection = pyodbc.connect(self.connection_string, attrs_before=attrs_before)
            cursor = connection.cursor()
            
            cursor.execute(query)
            
            # Obter nomes das colunas
            columns = [column[0] for column in cursor.description] if cursor.description else []
            
            # Obter dados
            rows = cursor.fetchall()
            
            # Converter dados para formato JSON serializável
            data = []
            for row in rows:
                row_dict = {}
                for i, column_name in enumerate(columns):
                    row_dict[column_name] = self._convert_value(row[i])
                data.append(row_dict)
            
            result = {
                "success": True,
                "row_count": len(data),
                "columns": columns,
                "data": data
            }
            
            return result
            
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "query": query
            }
        finally:
            if cursor:
                cursor.close()
            if connection:
                connection.close()

    @kernel_function(description="Obtém os metadados das tabelas do Lakehouse (esquema das tabelas).")
    def get_table_metadata(self) -> Annotated[str, "Retorna metadados das tabelas em formato JSON"]:
        """Obtém metadados das tabelas do Lakehouse."""
        if self._metadata_cache:
            return json.dumps(self._metadata_cache, ensure_ascii=False, indent=2)
        
        try:
            # Query para obter metadados das tabelas
            metadata_query = """
            SELECT 
                t.name AS TableName,
                c.name AS ColumnName,
                tp.name AS DataType,
                c.max_length,
                c.is_nullable
            FROM 
                sys.tables AS t
            INNER JOIN 
                sys.columns AS c ON t.object_id = c.object_id
            INNER JOIN 
                sys.types AS tp ON c.user_type_id = tp.user_type_id
            WHERE t.name NOT LIKE 'sys%'
            ORDER BY 
                t.name, c.column_id;
            """
            
            result = self._execute_query_pyodbc(metadata_query)
            
            if not result["success"]:
                return f"Erro ao obter metadados: {result['error']}"
            
            # Organizar metadados por tabela
            metadata = {}
            for row in result["data"]:
                table_name = row["TableName"]
                if table_name not in metadata:
                    metadata[table_name] = []
                
                metadata[table_name].append({
                    "column_name": row["ColumnName"],
                    "data_type": row["DataType"],
                    "max_length": row["max_length"],
                    "is_nullable": row["is_nullable"]
                })
            
            self._metadata_cache = metadata
            return json.dumps(metadata, ensure_ascii=False, indent=2)
            
        except Exception as e:
            return f"Erro ao obter metadados: {str(e)}"

    @kernel_function(description="Executa uma query SQL no Lakehouse e retorna os resultados.")
    def execute_query(self, sql_query: Annotated[str, "Query SQL para executar"]) -> Annotated[str, "Resultados da query em formato JSON"]:
        """Executa uma query SQL no Lakehouse."""
        try:
            result = self._execute_query_pyodbc(sql_query)
            return json.dumps(result, ensure_ascii=False, indent=2, default=str)
            
        except Exception as e:
            error_result = {
                "success": False,
                "error": str(e),
                "query": sql_query
            }
            return json.dumps(error_result, ensure_ascii=False, indent=2)

    @kernel_function(description="Lista as tabelas disponíveis no Lakehouse.")
    def list_tables(self) -> Annotated[str, "Lista de tabelas disponíveis"]:
        """Lista todas as tabelas disponíveis no Lakehouse."""
        try:
            query = "SELECT name FROM sys.tables WHERE name NOT LIKE 'sys%' ORDER BY name"
            result = self._execute_query_pyodbc(query)
            
            if not result["success"]:
                return f"Erro ao listar tabelas: {result['error']}"
            
            tables = [row["name"] for row in result["data"]]
            return json.dumps({"tables": tables}, ensure_ascii=False, indent=2)
            
        except Exception as e:
            return f"Erro ao listar tabelas: {str(e)}"

    @kernel_function(description="Executa uma query de contagem para validar estrutura da tabela.")
    def validate_table(self, table_name: Annotated[str, "Nome da tabela para validar"]) -> Annotated[str, "Resultado da validação em formato JSON"]:
        """Valida se uma tabela existe e retorna informações básicas."""
        try:
            query = f"SELECT COUNT(*) as total_rows FROM {table_name}"
            result = self._execute_query_pyodbc(query)
            
            if result["success"]:
                row_count = result["data"][0]["total_rows"]
                validation_result = {
                    "table_exists": True,
                    "table_name": table_name,
                    "total_rows": row_count
                }
            else:
                validation_result = {
                    "table_exists": False,
                    "table_name": table_name,
                    "error": result["error"]
                }
            
            return json.dumps(validation_result, ensure_ascii=False, indent=2)
            
        except Exception as e:
            error_result = {
                "table_exists": False,
                "table_name": table_name,
                "error": str(e)
            }
            return json.dumps(error_result, ensure_ascii=False, indent=2)

## Sistema Multi-Agente com AgentGroupChat

Baseado nos padrões do Semantic Kernel, vamos criar um sistema de três agentes especializados que colaboram usando AgentGroupChat:

1. **QueryGenerator** - Analisa metadados e gera queries T-SQL
2. **QueryExecutor** - Executa queries e valida resultados  
3. **ResultsAnalyst** - Analisa dados e gera relatórios executivos

O sistema utiliza uma estratégia de seleção sequencial fixa para garantir o fluxo correto: Geração → Execução → Análise.

In [ ]:
from semantic_kernel.agents import AgentGroupChat
from semantic_kernel.agents.strategies import SequentialSelectionStrategy, DefaultTerminationStrategy
from typing import List
from semantic_kernel.agents.strategies.selection.selection_strategy import SelectionStrategy
# Obs.: System Prompts Gerados por IA para Facilitar a vida.

# Instanciar o plugin
fabric_plugin = FabricDatabasePlugin()

# Agente 1: Gerador de Queries T-SQL baseado em metadados
query_generator = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="QueryGenerator",
    instructions="""Você é um especialista em geração de queries T-SQL para Microsoft Fabric Lakehouse.

# Responsabilidades Principais
1. Analisar perguntas do usuário e identificar requisitos de dados
2. SEMPRE chamar get_table_metadata() PRIMEIRO para obter esquema das tabelas
3. Analisar estrutura das tabelas e relacionamentos
4. Gerar query T-SQL otimizada que responda à pergunta

# Fluxo de Trabalho Obrigatório
1. Receber pergunta do usuário
2. Chamar get_table_metadata() para obter estrutura das tabelas
3. Identificar tabelas e colunas relevantes
4. Analisar relacionamentos (chaves primárias/estrangeiras)
5. Gerar query T-SQL otimizada

# Diretrizes Técnicas
- Use JOINs apropriados baseados nas relações entre tabelas
- Para vendas: factinternetsales, factresellersales
- Para produtos: dimproduct, dimproductcategory, dimproductsubcategory
- Para clientes: dimcustomer, dimgeography
- Para datas: dimdate com relacionamentos via DateKey
- Use aliases claros e consistentes
- Considere performance com TOP quando apropriado

# Formato de Saída
Retorne APENAS a query SQL pura, sem markdown, sem explicações, apenas o código T-SQL.""",
    plugins=[fabric_plugin],
)

# Agente 2: Executor e Validador de Queries
query_executor = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="QueryExecutor", 
    instructions="""Você é responsável por executar queries SQL e validar resultados.

# Responsabilidades Principais
1. Receber query SQL do QueryGenerator
2. Executar query usando execute_query()
3. Validar se execução foi bem-sucedida
4. Verificar consistência dos resultados
5. Preparar dados para análise

# Fluxo de Trabalho
1. Extrair query SQL da mensagem anterior
2. Executar usando execute_query()
3. Verificar se houve erros
4. Validar se resultados fazem sentido
5. Preparar dados estruturados para o analista

# Validações a Realizar
- Verificar se query executou sem erros SQL
- Confirmar se retornou dados (não vazio)
- Verificar consistência de tipos de dados
- Identificar valores nulos ou anômalos
- Validar se resultados são coerentes com a pergunta

# Formato de Saída
Se sucesso: JSON estruturado com os dados
Se erro: Detalhes do erro e recomendações de correção""",
    plugins=[fabric_plugin],
)

# Agente 3: Analista de Resultados e Gerador de Relatórios
results_analyst = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="ResultsAnalyst",
    instructions="""Você é um analista de dados especialista em gerar relatórios executivos.

# Responsabilidades Principais
1. Receber dados estruturados do QueryExecutor
2. Analisar resultados numericamente e contextualmente
3. Identificar insights, padrões e tendências
4. Gerar relatório executivo completo
5. Responder à pergunta original de forma clara

# Tipos de Análises
- Estatísticas descritivas (totais, médias, máximos, mínimos)
- Identificação de padrões e tendências
- Comparações entre segmentos
- Insights de negócio relevantes
- Recomendações baseadas nos dados

# Estrutura do Relatório Executivo
## 📊 Resumo Executivo
- Resposta direta à pergunta original
- Principais descobertas em bullet points

## 📈 Análise Detalhada  
- Estatísticas-chave com contexto
- Insights relevantes identificados
- Tendências e padrões observados

## 📋 Dados de Suporte
- Tabela formatada com resultados principais
- Números específicos e percentuais

## 💡 Recomendações
- Ações sugeridas baseadas nos dados
- Considerações estratégicas importantes

Use linguagem clara para stakeholders de negócio. Sempre inclua números específicos e contextualize os resultados.""",
)

print("✅ Agentes especializados criados:")
print("   • QueryGenerator - Gera queries T-SQL baseado em metadados")
print("   • QueryExecutor - Executa e valida queries") 
print("   • ResultsAnalyst - Analisa dados e gera relatórios")

✅ Agentes especializados criados:
   • QueryGenerator - Gera queries T-SQL baseado em metadados
   • QueryExecutor - Executa e valida queries
   • ResultsAnalyst - Analisa dados e gera relatórios


## Estratégia de Seleção Sequencial

Como este exemplo é de um cenário simples a estratégia sequencia ja resolve boa parte do estudo de caso. 

In [ ]:
# Cria um grupo de chat com os agentes
analyst_team = AgentGroupChat(
    agents=[query_generator, query_executor, results_analyst],
    # Especifica a estratégia de seleção sequencial e quem inicia
    selection_strategy=SequentialSelectionStrategy(initial_agent=query_generator),
    # limita o número máximo de iterações para 6 (2 turnos x 3 agentes)
    termination_strategy=DefaultTerminationStrategy(maximum_iterations=6),
)

print("Grupo de chat criado com os agentes: QueryGenerator, QueryExecutor, ResultsAnalyst")
print(f"Agente Inicial: {analyst_team.selection_strategy.initial_agent.name}")
print(f"Iterações máximas: {analyst_team.termination_strategy.maximum_iterations}")

Writing team chat created with sequential selection strategy
Initial agent: QueryGenerator
Maximum iterations: 6


In [ ]:
async def run_group_chat(group_chat: AgentGroupChat, question: str):
    
    await group_chat.add_chat_message(question)
    
    async for response in group_chat.invoke():
        print(f"*** {response.role} - {response.name or '*'}: '{response.content}'")
    print(f"*** History length: {len(group_chat.history)}")
    
    # Podemos redefinir manualmente o estado para manter o chat ativo
    # simple_group_chat.is_complete = False
    return group_chat.history
    
# testando o time analitico com uma pergunta
analyst_task = "Qual o total de vendas por ano e categoria de produto?"

# Executando o group chat
chat_history = await run_group_chat(analyst_team, analyst_task)

*** AuthorRole.ASSISTANT - QueryGenerator: 'SELECT
    d.CalendarYear AS Ano,
    pc.EnglishProductCategoryName AS CategoriaProduto,
    SUM(fis.SalesAmount) AS TotalVendas
FROM factinternetsales fis
JOIN dimdate d ON fis.OrderDateKey = d.DateKey
JOIN dimproduct p ON fis.ProductKey = p.ProductKey
JOIN dimproductsubcategory psc ON p.ProductSubcategoryKey = psc.ProductSubcategoryKey
JOIN dimproductcategory pc ON psc.ProductCategoryKey = pc.ProductCategoryKey
GROUP BY d.CalendarYear, pc.EnglishProductCategoryName
ORDER BY d.CalendarYear, pc.EnglishProductCategoryName;'
*** AuthorRole.ASSISTANT - QueryExecutor: '{
  "resultado": [
    {
      "Ano": 2010,
      "CategoriaProduto": "Bikes",
      "TotalVendas": 43421.04
    },
    {
      "Ano": 2011,
      "CategoriaProduto": "Bikes",
      "TotalVendas": 7075526.38
    },
    {
      "Ano": 2012,
      "CategoriaProduto": "Accessories",
      "TotalVendas": 2147.08
    },
    {
      "Ano": 2012,
      "CategoriaProduto": "Bikes",
      "

# Conclusão Parte 2

Vimos como criar um grupo de agentes simples para poder chamar plugins e executar queries dentro do MS Fabric, usando o PyODBC e Semantic Kernel.    
O mesmo Conceito funcionaria para uso de outras bibliotecas como Pydantic AI e Langgraph. Bastando apenas algumas modificações nos codigos.   
O arquivo : [multi_agent_output_sample](MS-Fabric-Workshop\assets\notebooks\aigents\multi_agent_output_samplec.md) Contem um exemplo de saida.  